# 10 — Medical Tokenizer & Pretraining Data

This notebook builds the pretraining corpus and trains a **medical-domain BPE tokenizer**.

**Pretraining corpus (two sources combined):**
1. `epfl-llm/guidelines` — 33k clinical guidelines (WHO, CDC, NICE)
2. `pubmed_qa` pqa_unlabeled — 60k biomedical abstracts with context paragraphs

**Pipeline:**
1. Load both datasets (no streaming needed — both fit in memory)
2. Combine and save train / val / test text splits
3. Train a new BPE tokenizer on the combined medical training text
4. Compare it to the TinyStories tokenizer on 10 benchmark medical terms
5. Tokenize all splits into binary `.npy` files for training

**Why two sources?**  
Guidelines give long-form structured clinical prose (~7k words/doc).  
PubMedQA gives concise biomedical abstract language (~300 words/doc).  
Together they cover a wider range of medical writing styles.


## 1 — Setup

In [3]:
# ── Environment setup ─────────────────────────────────────────────────────
import os, sys

if os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

sys.path.insert(0, os.path.abspath(".."))

import config as cfg
cfg.make_dirs()
cfg.print_config()

  slm-from-scratch config
  Environment : Local
  Base path   : /Users/aman2394/Desktop/slm-learning
  --- Model ---
  vocab_size  : 8192
  block_size  : 256
  n_layer     : 8  |  n_head: 8  |  n_embd: 512
  --- Pretraining ---
  lr          : 0.0003  |  steps: 20000
  batch_size  : 16  |  grad_accum: 1
  precision   : fp16
  --- SFT ---
  lr          : 3e-05  |  steps: 3000


In [4]:
# !pip install datasets tokenizers tqdm
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.10.0
CUDA available: False


## 2 — Load & Save Pretraining Corpus

We load both datasets fully into memory (no streaming — each is under 1GB).
Documents from both sources are combined before splitting into train / val / test.

**Text extraction:**
- Guidelines: `clean_text` field (full guideline markdown)
- PubMedQA unlabeled: `question` + `context["contexts"]` paragraphs + `long_answer`

Documents are shuffled before splitting so val/test contain a mix of both sources.

**Split sizes** (from `config.py`):
- Train: `cfg.MED_TEXTBOOK_DATA_SIZE` (33k) guidelines + `cfg.MED_PUBMEDQA_PRETRAIN_SIZE` (60k) PubMedQA
- Val / Test: `cfg.MED_VAL_SIZE` + `cfg.MED_TEST_SIZE` drawn from guidelines remainder

**Expected time:** ~2–5 min (download + processing).


In [5]:
from datasets import load_dataset
from tqdm import tqdm
import random
import config as cfg

# ── Load Dataset 1: epfl-llm/guidelines ──────────────────────────────────────
print("Loading epfl-llm/guidelines...")
guidelines = load_dataset(cfg.MED_TEXTBOOK_DATASET, split="train")
print(f"  {len(guidelines):,} examples")

# ── Load Dataset 2: PubMedQA unlabeled ───────────────────────────────────────
print("Loading pubmed_qa pqa_unlabeled...")
pubmedqa_ul = load_dataset(
    cfg.MED_PUBMEDQA_DATASET,
    cfg.MED_PUBMEDQA_PRETRAIN_SUBSET,
    split="train",
)
print(f"  {len(pubmedqa_ul):,} examples")

# ── Extract text from each source ────────────────────────────────────────────
def extract_guidelines_text(ex):
    """Return the clean text of a guidelines document."""
    return ex["clean_text"].strip()

def extract_pubmedqa_text(ex):
    """Concatenate question + context paragraphs + long_answer into one document."""
    parts = [ex["question"]] + ex["context"]["contexts"]
    if ex.get("long_answer"):
        parts.append(ex["long_answer"])
    return " ".join(parts).strip()

# ── Extract all texts ────────────────────────────────────────────────────────
guidelines_texts = [extract_guidelines_text(ex) for ex in tqdm(guidelines, desc="Guidelines")]
pubmedqa_texts   = [
    extract_pubmedqa_text(pubmedqa_ul[i])
    for i in tqdm(range(min(cfg.MED_PUBMEDQA_PRETRAIN_SIZE, len(pubmedqa_ul))), desc="PubMedQA")
]

# ── Split each source into train / val / test ─────────────────────────────────
# Val and test each get 1000 docs from guidelines + 1000 from pubmedqa.
# This means validation loss reflects the full training distribution,
# not just guidelines text.
VAL_PER_SOURCE  = cfg.MED_VAL_SIZE  // 2   # 1000 each
TEST_PER_SOURCE = cfg.MED_TEST_SIZE // 2   # 1000 each

# Guidelines: first MED_TEXTBOOK_DATA_SIZE for train, remainder for val+test
gl_train = guidelines_texts[:cfg.MED_TEXTBOOK_DATA_SIZE]
gl_val   = guidelines_texts[cfg.MED_TEXTBOOK_DATA_SIZE : cfg.MED_TEXTBOOK_DATA_SIZE + VAL_PER_SOURCE]
gl_test  = guidelines_texts[cfg.MED_TEXTBOOK_DATA_SIZE + VAL_PER_SOURCE : cfg.MED_TEXTBOOK_DATA_SIZE + VAL_PER_SOURCE + TEST_PER_SOURCE]

# PubMedQA: last 2000 reserved for val+test, rest for train
pm_train = pubmedqa_texts[:-(VAL_PER_SOURCE + TEST_PER_SOURCE)]
pm_val   = pubmedqa_texts[-(VAL_PER_SOURCE + TEST_PER_SOURCE) : -TEST_PER_SOURCE]
pm_test  = pubmedqa_texts[-TEST_PER_SOURCE:]

# Combine and shuffle each split
train_texts = gl_train + pm_train
val_texts   = gl_val   + pm_val
test_texts  = gl_test  + pm_test

random.shuffle(train_texts)
random.shuffle(val_texts)
random.shuffle(test_texts)

print(f"\nCorpus sizes:")
print(f"  train : {len(train_texts):,}  ({len(gl_train):,} guidelines + {len(pm_train):,} pubmedqa)")
print(f"  val   : {len(val_texts):,}   ({len(gl_val):,} guidelines + {len(pm_val):,} pubmedqa)")
print(f"  test  : {len(test_texts):,}   ({len(gl_test):,} guidelines + {len(pm_test):,} pubmedqa)")


/Users/aman2394/Desktop/slm-learning/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading epfl-llm/guidelines...


  37,970 examples
Loading pubmed_qa pqa_unlabeled...
  61,249 examples


PubMedQA: 100%|██████████| 60000/60000 [00:02<00:00, 27820.31it/s]



Corpus sizes:
  train : 91,000  (33,000 guidelines + 58,000 pubmedqa)
  val   : 2,000   (1,000 guidelines + 1,000 pubmedqa)
  test  : 2,000   (1,000 guidelines + 1,000 pubmedqa)


In [ ]:
from tokenizer.preprocess import train_tokenizer
import config as cfg

# Train on BOTH files:
#   1. MED_TRAIN_TXT        — pretraining corpus (guidelines + pubmedqa_unlabeled)
# The tokenizer sees all medical vocabulary that will appear during both
# pretraining and SFT, so it can form efficient merges for medical terms.
med_tok = train_tokenizer(
    train_file=[cfg.MED_TRAIN_TXT, MED_TOKENIZER_EXTRA_TXT],
    save_dir=cfg.MED_TOKENIZER_DIR,
    vocab_size=cfg.VOCAB_SIZE,
)

print(f"Vocab size : {med_tok.get_vocab_size()}")
print(f"EOS id     : {med_tok.token_to_id('</s>')}")
print(f"PAD id     : {med_tok.token_to_id('<pad>')}")


## 2b — Save PubMedQA Labeled Text for Tokenizer Training

The `pqa_labeled` split is used for SFT, but its vocabulary (medical questions,
context paragraphs, yes/no/maybe answers) should also be seen by the tokenizer
during training. Otherwise, SFT data may contain word fragments the tokenizer
never merged efficiently.

We save the labeled text to a separate file and pass **both** files to
`train_tokenizer()`. This does not mix SFT data into the pretraining corpus —
it only informs the BPE merge rules.


In [ ]:
from datasets import load_dataset
import config as cfg
import os

# Path for pqa_labeled text used only for tokenizer training
MED_TOKENIZER_EXTRA_TXT = os.path.join(cfg.MED_DATA_DIR, "tokenizer_extra.txt")

print("Loading pubmed_qa pqa_labeled...")
pqa_labeled = load_dataset(cfg.MED_PUBMEDQA_DATASET, "pqa_labeled", split="train")
print(f"  {len(pqa_labeled):,} examples")

def extract_pqa_labeled_text(ex):
    """Combine question + context paragraphs + long_answer into one document."""
    parts = [ex["question"]] + ex["context"]["contexts"]
    if ex.get("long_answer"):
        parts.append(ex["long_answer"])
    return " ".join(parts).strip()

pqa_labeled_texts = [extract_pqa_labeled_text(ex) for ex in pqa_labeled]
save_split(pqa_labeled_texts, MED_TOKENIZER_EXTRA_TXT)
print(f"Saved {len(pqa_labeled_texts):,} pqa_labeled documents → {MED_TOKENIZER_EXTRA_TXT}")


## 3 — Train Medical BPE Tokenizer

`train_tokenizer()` now accepts a list of files, so we pass **both**:
- `MED_TRAIN_TXT` — the pretraining corpus (guidelines + pubmedqa_unlabeled)
- `MED_TOKENIZER_EXTRA_TXT` — the pqa_labeled SFT text

This ensures the tokenizer learns efficient merges for vocabulary that will
appear during **both** pretraining and SFT. Without this, medical terms in
the SFT data might get split into more subwords than necessary, wasting
sequence length and making training harder.

We reuse `cfg.VOCAB_SIZE` (8192) — keeping architecture compatible.

**Expected time:** ~5 minutes.


In [ ]:
from tokenizer.preprocess import train_tokenizer
import config as cfg

print(f"Training BPE tokenizer on: {cfg.MED_TRAIN_TXT}")
print(f"Target vocab size         : {cfg.VOCAB_SIZE:,}")
print(f"Output directory          : {cfg.MED_TOKENIZER_DIR}")

med_tok = train_tokenizer(
    train_file=cfg.MED_TRAIN_TXT,
    save_dir=cfg.MED_TOKENIZER_DIR,
    vocab_size=cfg.VOCAB_SIZE,   # same size as TinyStories — keeps architecture compatible
)

print(f"\nTokenizer trained.")
print(f"  Vocab size : {med_tok.get_vocab_size():,}")
print(f"  EOS token  : {med_tok.token_to_id('</s>')}")

## 4 — Tokenizer Comparison

We compare the TinyStories tokenizer against the new medical tokenizer on the same 10 benchmark
medical terms from notebook 10.

A good medical tokenizer should need **fewer tokens** for each term — this directly reduces
sequence length and improves the model's ability to reason about medical concepts as units.

In [ ]:
from tokenizer.preprocess import load_tokenizer
import config as cfg

ts_tok = load_tokenizer(cfg.TOKENIZER_VOCAB, cfg.TOKENIZER_MERGES)
med_tok   = load_tokenizer(cfg.MED_TOKENIZER_VOCAB, cfg.MED_TOKENIZER_MERGES)

MEDICAL_TERMS = [
    "myocardial infarction",
    "hypertension",
    "pneumonia",
    "pharmacokinetics",
    "tachycardia",
    "stethoscope",
    "anesthesia",
    "oncology",
    "contraindication",
    "ventricular",
]

print(f"{'Term':<30} {'Cosmo #tok':>12} {'Med #tok':>10}  Improvement")
print("-" * 70)
total_cosmo = total_med = 0
for term in MEDICAL_TERMS:
    c_n = len(ts_tok.encode(term).ids)
    m_n = len(med_tok.encode(term).ids)
    delta = c_n - m_n
    flag = f"↓ {delta}" if delta > 0 else ("=" if delta == 0 else f"↑ {-delta}")
    print(f"{term:<30} {c_n:>12} {m_n:>10}  {flag}")
    total_cosmo += c_n
    total_med   += m_n

print("-" * 70)
print(f"{'TOTAL':<30} {total_cosmo:>12} {total_med:>10}  ↓ {total_cosmo - total_med} tokens total")
print(f"\nAverage compression: {total_cosmo / total_med:.2f}x")

## 5 — Tokenize All Splits

`tokenize_and_save()` from `tokenizer/preprocess.py` reads lines from the `.txt` file, encodes
them with the given tokenizer, appends an EOS token after each document, and concatenates everything
into a flat `uint16` numpy array saved as a `.npy` file.

**Why flat `uint16`?**  
- A flat array allows random-access indexing: `data[i : i + block_size]` is a O(1) operation
- `uint16` stores values 0–65535, which covers any vocab up to 65k tokens
- Memory-mapped with `np.memmap` — the OS pages in only the slices we access

We temporarily patch the config paths so `tokenize_and_save` writes to the medical paths
rather than the default TinyStories paths.

**Expected time:** 20–40 minutes for 80k documents.

In [ ]:
import config as cfg
from tokenizer.preprocess import load_tokenizer, tokenize_and_save

# Load the freshly trained medical tokenizer
med_tok = load_tokenizer(cfg.MED_TOKENIZER_VOCAB, cfg.MED_TOKENIZER_MERGES)

# ── Temporarily patch config so tokenize_and_save uses medical paths ─────
# tokenize_and_save() reads cfg.TRAIN_TXT and writes cfg.TRAIN_TOKENS, etc.
# We save the originals and restore them after the call.
_orig = {
    "TRAIN_TXT":    cfg.TRAIN_TXT,
    "VAL_TXT":      cfg.VAL_TXT,
    "TEST_TXT":     cfg.TEST_TXT,
    "TRAIN_TOKENS": cfg.TRAIN_TOKENS,
    "VAL_TOKENS":   cfg.VAL_TOKENS,
    "TEST_TOKENS":  cfg.TEST_TOKENS,
}

cfg.TRAIN_TXT    = cfg.MED_TRAIN_TXT
cfg.VAL_TXT      = cfg.MED_VAL_TXT
cfg.TEST_TXT     = cfg.MED_TEST_TXT
cfg.TRAIN_TOKENS = cfg.MED_TRAIN_TOKENS
cfg.VAL_TOKENS   = cfg.MED_VAL_TOKENS
cfg.TEST_TOKENS  = cfg.MED_TEST_TOKENS

print("Tokenizing train split...")
tokenize_and_save("train", med_tok, batch_size=4000)

print("Tokenizing val split...")
tokenize_and_save("val", med_tok, batch_size=4000)

print("Tokenizing test split...")
tokenize_and_save("test", med_tok, batch_size=4000)

# ── Restore original config paths ─────────────────────────────────────────
for k, v in _orig.items():
    setattr(cfg, k, v)

print("\nAll splits tokenized. Config paths restored.")

## 6 — Verify

Load each `.npy` file with `np.memmap` and print the token count.
This confirms all three files were written successfully and have reasonable sizes.

In [ ]:
import numpy as np
import config as cfg

splits = [
    ("train", cfg.MED_TRAIN_TOKENS),
    ("val",   cfg.MED_VAL_TOKENS),
    ("test",  cfg.MED_TEST_TOKENS),
]

print(f"{'Split':<10} {'Tokens':>15}  {'Millions':>10}  File")
print("-" * 70)
for name, path in splits:
    import os
    if not os.path.exists(path):
        print(f"  {name:<10} FILE NOT FOUND: {path}")
        continue
    arr = np.memmap(path, dtype=np.uint16, mode="r")
    print(f"  {name:<10} {len(arr):>15,}  {len(arr)/1e6:>10.1f}M  {path}")

print()
print("Verify that train >> val ≈ test (by the configured size ratios).")
print("If any file is 0 bytes or missing, re-run the tokenize_and_save cells above.")

## Next Steps

The medical corpus is ready. Next:

- **Notebook 12** — Domain-Adaptive Pretraining: load the TinyStories checkpoint, swap the
  embedding table for the new medical vocabulary, and continue training on the medical corpus.
- The tokenizer files in `cfg.MED_TOKENIZER_DIR` will be used by every subsequent notebook.

To push the tokenizer to HuggingFace Hub now (optional):
```python
from huggingface_hub import HfApi
HfApi().upload_folder(
    folder_path=cfg.MED_TOKENIZER_DIR,
    repo_id=cfg.MED_HF_TOKENIZER_REPO,
    repo_type="model",
)
```